# 🎯 [K12_M04] FAQ 근거 답변 확인하기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K12/k12_m04_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 일반행정·부대관리 (Level 1) / K12 행정 업무 FAQ 챗봇 설계 |
| **미션** | 미션 04. FAQ 근거 답변 확인하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 생성형 AI 시스템 프롬프트 엔지니어링 및 팩트체크 검증 워크북 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 데이터셋 로더
# ============================================================================
from IPython.display import display, Markdown
import json
import os

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드하여 반환합니다."""
    filename = f"k12_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K12", "../uploads/K12"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_dataset_summary(mission_num):
    """해당 미션의 데이터셋 주요 항목을 요약 출력합니다."""
    data = load_dataset(mission_num)
    if data:
        display(Markdown(f"**[데이터셋 로드 성공]** `{data.get('title', '')}` ({data.get('mission', '')})"))
        keys = [k for k in data.keys() if k not in ["project", "mission", "title"]]
        print(f"포함 항목: {', '.join(keys)}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 답변 및 검증 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 생성 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 데이터셋 로더(k12_m01~m10) 준비 완료!")



---

## 🎯 미션 04. FAQ 근거 답변 확인하기
**소요시간**: 약 1.5시간

![미션 4 챗봇 생성 답변 단위 사실 1:1 대조 점검 및 출처 FAQ 번호 표기 체계 다이어그램](/uploads/common/K12_m04_factcheck_verification.png)

### 📋 미션 개요 및 배경
생성형 AI가 복잡한 행정 절차나 필요 서류를 임의로 왜곡·축약하여 발생할 수 있는 오안내 위험을 방지하기 위해 답변 대조 검증을 수행합니다. 챗봇이 생성한 답변의 처리 순서, 필수 서류, 접수 기한, 내선 번호를 원본 FAQ와 1:1로 비교하여 '맞음/틀림/지어냄'으로 판정하고 오류 유발 패턴을 분석합니다. 답변 끝에 인용된 FAQ 번호를 의무 표기하도록 프롬프트를 고도화하여 인간 검증(HITL)을 용이하게 하는 추적성 확보 기술을 익힙니다.

### 🎯 미션 학습목표
답변의 절차 순서·서류·기한·연락처를 FAQ와 하나씩 대조해 맞음·틀림·지어냄으로 표시합니다. 틀린 답은 어떤 질문 표현에서 생겼는지 기록하고, 답변에 근거 FAQ 번호를 쓰게 해 확인을 쉽게 만듭니다. **이를 통해 AI의 환각(Hallucination) 현상을 원문과 1:1로 검증하는 인간 검증(HITL) 습관과 인용 출처 기반의 답변 추적성 설계 기법을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 답변을 대조표에 맞음·틀림·지어냄으로 표시한다.
#### 이 단계의 역할

챗봇의 답변 속 절차 순서, 필요 서류, 접수 기한, 연락처를 원문 FAQ와 1:1 대조하여 [맞음 / 틀림 / 지어냄] 3단계로 판정합니다.

#### 해야 할 일

- 5개 우선 질문에 대한 챗봇 답변을 문장 단위로 분해합니다.

- 각 문장의 사실 정보(기한 숫자, 서류 명칭, 담당 부서)를 FAQ 원문 표와 교차 검증합니다.

- 검증 결과를 [맞음 / 틀림(정보 왜곡) / 지어냄(없는 정보 생성)] 3가지 척도로 대조표에 기록합니다.

#### 💡 AI 프롬프트 질의 예시

> **[답변 대조 및 팩트체크 보조 질의]:** "아래 [FAQ 원문 항목]과 [챗봇 생성 답변]을 비교하여 1) 처리 절차, 2) 구비 서류, 3) 접수 기한, 4) 담당 부서가 원문과 100% 일치하는지 대조 분석표를 작성하고 불일치 요소를 지적해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **숫자·명칭 정밀 대조:** 기한(D-3 등)과 서식 명칭이 글자 하나까지 원문과 일치하는가?
- **지어냄(할루시네이션) 감지:** FAQ 원문에 없는 불필요한 절차나 조건이 임의로 덧붙여지지 않았는가?

#### 핵심 확인 지표

> 5개 질문에 대한 답변 원문 대조 검증표

#### 학습자가 제출해야 하는 내용

- 원문 대조 점검표 완성본

#### 🔹 Step 2. 틀린 답의 질문 표현을 기록한다.
#### 이 단계의 역할

오답이나 왜곡이 발생한 경우, 질문자의 질의 표현(모호한 단어, 축약어, 복합 질문 등) 중 어떤 요소가 AI의 오독을 유발했는지 원인을 분석합니다.

#### 해야 할 일

- 대조표에서 '틀림' 또는 '지어냄'으로 판정된 항목을 선별합니다.

- 사용자 질문에 사용된 어휘 중 다의어, 줄임말, 주어 생략 등 AI 혼선을 유발한 요인을 추출합니다.

- AI에게 오답 발생 메커니즘을 분석하도록 질의하여 질문 패턴별 취약점을 목록화합니다.

#### 💡 AI 프롬프트 질의 예시

> **[오답 유발 질문 표현 분석 질의]:** "챗봇이 이 질문에서 절차를 누락하거나 서류를 잘못 안내한 원인을 분석해 줘. 질문자의 어떤 어휘나 문장 구조가 AI의 검색 오류를 유발했는지 2가지 원인을 짚어줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **원인 진단의 구체성:** 단순 'AI 실수'가 아니라 어휘 매칭 실패나 문맥 오해 등 구체적 원인이 규명되었는가?

#### 핵심 확인 지표

> 오류 유발 질문 표현 분석서

#### 학습자가 제출해야 하는 내용

- 오류 원인 분석 메모

#### 🔹 Step 3. 답변 끝에 근거 FAQ 번호를 쓰게 한다.
#### 이 단계의 역할

답변의 신뢰성과 추적성을 확보하기 위해, 답변 말미에 반드시 근거가 된 FAQ 항목 번호(예: [근거: FAQ-F03])를 출력하도록 프롬프트를 개선합니다.

#### 해야 할 일

- 시스템 프롬프트의 답변 규칙에 "모든 답변의 맨 끝 줄에 반드시 [근거: FAQ-F번호]를 명시하라"는 강제 지시를 추가합니다.

- 동일한 5개 질문을 다시 질의하여 근거 번호가 정확히 출력되는지 확인합니다.

- 답변에 표기된 근거 번호가 실제 해당 내용을 담고 있는 FAQ 원문 번호와 일치하는지 대조합니다.

#### 💡 AI 프롬프트 질의 예시

> **[출처 인용 강제 프롬프트 개선 질의]:** "챗봇이 모든 답변의 맨 마지막 줄에 반드시 참조한 FAQ의 번호를 '[근거: FAQ-F번호]' 형태로 명시하도록 시스템 지시문에 추가할 엄격한 명령 문구를 작성해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **인용 번호 정확성:** 답변 끝에 달린 번호(예: F03)를 확인했을 때 실제 해당 행정 절차가 맞는지 원문 대조
- **서식 준수:** 대괄호 및 양식 표기가 일관되게 출력되는가?

#### 핵심 확인 지표

> 근거 번호 출력 규칙이 반영된 개선 프롬프트 및 질의 결과

#### 학습자가 제출해야 하는 내용

- 근거 번호가 포함된 답변 결과 로그

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할

복합 질문(2개 이상의 절차가 얽힌 질의)에 대해 2개 이상의 근거 번호(예: [근거: FAQ-F02, F05])를 동시에 정확히 표기하는 규칙을 테스트합니다.

#### 해야 할 일

- "휴가 신청하면서 군 경력증명서도 같이 뗄 수 있나요?"와 같은 복합 질의를 생성합니다.

- AI가 다중 출처를 정확히 병기하는지 확인하고 프롬프트를 보강합니다.

#### 💡 AI 프롬프트 질의 예시

> **[다중 출처 인용 테스트]:** "복수 행정 절차 문의 시 관련 FAQ 번호를 모두 쉼표로 연결하여 표기하도록 시스템 규칙을 테스트해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **다중 출처 정확도:** 언급된 두 절차의 FAQ 번호가 모두 올바른가?

#### 핵심 확인 지표

> 다중 출처 표기 검증 메모

#### 학습자가 제출해야 하는 내용

- 심화 도전 과제 메모

### 📦 미션 필수 학습자료 및 팁
**미션 수행 필수 학습자료**

- **단위 행정 사실 교차 검증 대조표 양식 (Fact-checking Sheet):** 절차 순서, 필요 서류, 접수 기한, 연락처별 '맞음/틀림/지어냄' 1:1 판정 서식
- **근거 FAQ 번호 표기 의무화 프롬프트 템플릿:** 답변 말미에 [근거: FAQ-F번호]를 의무 출력하도록 제약 조건이 보강된 프롬프트
- **[학습 가이드] AI 환각(Hallucination) 식별 및 인간 검증(HITL) 실무 수칙:** 행정 서류 누락이나 기한 왜곡 등 AI의 전형적인 오류 패턴을 포착하고 대조하는 육안 검증 가이드

**실습 팁:** 날짜 숫자(예: 3일 전 vs 5일 전)나 서류 명칭(예: 휴가신청서 vs 휴가계획서)은 AI가 그럴듯하게 왜곡하기 쉬우므로, FAQ 원문과 글자 하나까지 대조 검증하세요.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] AI 결과 검증 — 답변 속 사실 정보를 FAQ 원문과 1:1 대조하여 팩트체크하고, 근거 번호 표기 규칙을 성공적으로 반영했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 5개 문항 단위 사실 원문 대조 점검표 완성 (맞음/틀림/지어냄 판정)

- 오답 유발 질문 표현 요인 분석 완료

- 출처(근거 FAQ 번호) 강제 표기 개선 프롬프트 및 질의 결과 확보

#### 채점자 판정 메모

> 답변 말미에 붙은 [근거: FAQ-F번호]가 실제 질문 내용과 일치하는 FAQ 번호인지 육안으로 대조 확인한다.



In [ ]:
# [미션 04.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_04_prompt = """
[챗봇 생성 답변 단위 사실 1:1 대조 점검 및 출처 표기 의무화]
다음 AI 챗봇이 생성한 정기휴가 신청 절차 안내문에 대해 원본 FAQ(F01)의 내용과 문장 단위로 1:1 교차 대조(Fact-checking)를 수행해 주십시오.
각 문장별로 [일치(맞음) / 사실 왜곡(틀림) / 환각(지어냄)] 여부를 판정하고 수정 권고안을 제시해 주십시오.
또한 향후 AI가 자의적으로 절차를 왜곡하지 못하도록 모든 답변 하단에 근거 FAQ 번호(예: [출처: FAQ-F01])를 의무적으로 표기하도록 강제하는 프롬프트 개선안을 도출해 주십시오.

[검증 대상 AI 생성 답변]:
"정기휴가는 출타 1일 전까지 행정반에 전화로 신청하시면 됩니다. 구두 신청 후 휴가 당일에 휴가신청서를 작성하여 중대장님께 직접 서명받으면 즉시 휴가증이 발급됩니다."
"""

mission_04_result = """
## 1. 단위 사실 원문 대조 점검표 (Fact-checking Sheet)

| 응답 번호 | 질문 요약 | AI 생성 답변의 핵심 주장 | FAQ 원문 내용 대조 (진위 확인) | 판정 (맞음/틀림/지어냄) | 오류 원인 분석 |
| :---: | :--- | :--- | :--- | :---: | :--- |
| **R01** | 포상휴가 기한/서류 | 6개월 이내, D-3일 전 제출, 인사과 | F02: 수여일 6개월 이내, D-3일 전, 포상명령 사본 | **맞음** | 원문 조건과 100% 일치 |
| **R02** | 복무확인서 발급 | 즉시 1시간 내 발급, 수수료 무료 | F04: 신청일 기준 24시간 이내 발급 (수수료 언급 없음) | **지어냄** | '1시간 내' 및 '수수료 무료'를 임의로 창작함 |
| **R03** | 자격증 응시료 | 연간 12만 원, 익월 10일까지 신청 | F10: 연간 최대 12만 원, 익월 10일까지 신청 | **맞음** | 금액 및 신청 기한 완벽 일치 |
| **R04** | 비밀번호 초기화 | 대리 신청 가능 | F12: 본인 신분증 지참 통신실 방문 (대리 불가) | **틀림** | 보안 원칙(본인 직접 방문)을 정반대로 왜곡 |
| **R05** | 전등 고장 수리 | 16시 취합 후 익일 수리, 군수과 | F14: 매일 16:00 취합 후 익일 수리 원칙, 군수과 | **맞음** | 수리 일정과 부서 완벽 일치 |

---

## 2. 출처(근거) 표기 의무화 프롬프트 개선안

```markdown
[추가 시스템 지시 규칙]
- [출처 표기 의무]: 모든 답변의 맨 마지막 줄에는 반드시 참조한 FAQ의 고유 번호를 아래 형식으로 출력하십시오.
  출력 형식 예시: [근거: FAQ-F02]
- 만약 복수의 항목을 참조했다면 쉼표로 연결하여 표기하십시오. 예: [근거: FAQ-F01, FAQ-F02]
```
"""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 04. FAQ 근거 답변 확인하기", mission_04_prompt, mission_04_result)

